In [4]:
#!uv add -U langchain-core langchain-classic langchain-community

In [ ]:
#!uv add -U langchain-openai langchain-chroma langchain-text-splitters

In [ ]:
#!uv add -U rank-bm25 faiss-cpu lark

In [3]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pathlib import Path

data_dir = Path("data/retriever")

documents = [
    Document(
        page_content=file_path.read_text(encoding='utf-8'),
        metadata={
            "source": file_path.name,
            "path": str(file_path.relative_to(data_dir)),
        },
    )
    for file_path in sorted(data_dir.glob("*"))
    if file_path.is_file()
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4.1-mini")
vector_store = InMemoryVectorStore.from_documents(documents, embeddings)

In [9]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":2, "fetch_k":4, "lambda_mult":0.7},
)

results=retriever.invoke("모터 이상 발생 시 점검 항목")
for document in results:
    print(document.metadata["source"], document.page_content)

11_compression_long_report.md # 모터 설비 월간 점검 보고서

## 생산 현황
이번 달 생산량은 계획 대비 98%였고 설비 가동률은 91%였습니다. 주간 생산 계획에는 큰 차이가 없었습니다.

## 청소 상태
설비 외관 청소 상태는 양호했습니다. 모터 주변 통로에 자재가 일부 적치되어 이동 조치했습니다.

## 진동 이상
9월 18일 오전부터 모터 진동이 평균 2.2 mm/s에서 6.1 mm/s까지 증가했습니다.
점검 결과 베어링 윤활 상태가 부족했고 축 정렬 오차도 발견되었습니다.
윤활 보충과 축 정렬 작업 후 진동은 2.5 mm/s 수준으로 감소했습니다.

## 온도
진동 증가 시간대에 베어링 온도도 58도에서 76도까지 상승했습니다.
조치 후 온도는 61도로 회복되었습니다.

## 전력
모터 전력 사용량은 전월 대비 2% 증가했으나 생산량 차이를 고려하면 유의한 변화는 아니었습니다.

## 안전
월간 안전 점검에서 비상 정지 버튼과 안전 커버의 이상은 발견되지 않았습니다.

## 향후 조치
진동과 온도가 동시에 증가할 경우 우선 점검 알람을 발생하도록 기준을 검토합니다.

09_timeweighted_documents.json [
  {
    "document_id": "T-001",
    "text": "모터 진동 증가 시 베어링 마모와 축 정렬을 점검합니다.",
    "event_time": "2026-09-01T09:00:00+09:00",
    "topic": "motor",
    "importance": "high"
  },
  {
    "document_id": "T-002",
    "text": "모터 베어링 온도가 75도를 초과해 윤활 상태를 점검했습니다.",
    "event_time": "2026-09-20T14:30:00+09:00",
    "topic": "motor",
    "importance": "high"
  },
  {
    "document_id": "T-003",
    "te

In [22]:
import csv
from pathlib import Path
from langchain_core.documents import Document

data_dir = Path("data/retriever")
target_files = ["01_common_documents.csv","03_keyword_bm25_documents.csv","04_reorder_documents.csv", "07_selfquery_documents.csv"]

documents=[]
for name in target_files:
    file_path = data_dir / name
    with file_path.open(encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            metadata = dict(row)
            metadata.pop("text", None)
            for key in ("year", "priority", "original_order"):
                if key in metadata:
                    metadata[key] = int(metadata[key])
            metadata["source"] = file_path.name
            documents.append(
                Document(
                    page_content=row["text"],
                    metadata=metadata
                )
            )

In [23]:
print(len(documents))
print(documents[0].metadata)
print(sum(1 for d in documents if d.metadata.get("equipment") == "motor"))

vector_store = InMemoryVectorStore.from_documents(documents, embeddings)
filtered_retriever = vector_store.as_retriever(
    search_kwargs={"k":2, "filter": lambda doc: doc.metadata.get("equipment") =="motor"},
)

results=filtered_retriever.invoke("모터 최근 점검 결과")

36
{'document_id': 'M-001', 'equipment': 'press', 'year': 2025, 'priority': 3, 'category': 'maintenance', 'plant': 'plant_a', 'source': '01_common_documents.csv'}
5


In [24]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

base_retriever = vector_store.as_retriever(search_kwargs={"k":4})
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor,
)

results = compression_retriever.invoke("모터 진동 원인과 조치")
for document in results:
    print(document.page_content)
    print(document.metadata)
    print("---")

모터 진동 증가 시 베어링 마모와 축 정렬을 우선 점검합니다.
{'document_id': 'R-003', 'original_order': 3, 'source': '04_reorder_documents.csv'}
---
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
{'document_id': 'SQ-003', 'equipment': 'motor', 'year': 2025, 'priority': 3, 'category': 'maintenance', 'plant': 'plant_a', 'source': '07_selfquery_documents.csv'}
---
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
{'document_id': 'M-002', 'equipment': 'motor', 'year': 2024, 'priority': 2, 'category': 'maintenance', 'plant': 'plant_a', 'source': '01_common_documents.csv'}
---
모터의 이름판과 정격 전압을 확인합니다.
{'document_id': 'R-004', 'original_order': 4, 'source': '04_reorder_documents.csv'}
---


In [25]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 3
vector_retriever = vector_store.as_retriever(search_kwargs={"k":3})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],
)

results = ensemble_retriever.invoke("M-002 모터 축 정렬")
for document in results:
    print(document.metadata["document_id"], "|", document.metadata["source"])
    print(document.page_content)
    print("---")

M-002 | 01_common_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
---
R-003 | 04_reorder_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬을 우선 점검합니다.
---


In [30]:
from langchain_community.document_transformers import LongContextReorder

retriever = vector_store.as_retriever(search_kwargs={"k":4})
retrieved_documents = retriever.invoke("모터 점검과 이상 조치")

reordering = LongContextReorder()
reordered_documents = reordering.transform_documents(retrieved_documents)

for index, document in enumerate(reordered_documents, start=1):
    print(index, document.metadata["document_id"], "|", document.metadata["source"])
    print(document.page_content)

context = "\n\n".join(
    document.page_content for document in reordered_documents
)

1 SQ-003 | 07_selfquery_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
2 R-004 | 04_reorder_documents.csv
모터의 이름판과 정격 전압을 확인합니다.
3 M-002 | 01_common_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
4 R-003 | 04_reorder_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬을 우선 점검합니다.


In [58]:
from langchain_chroma import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

md_paths = list(data_dir.glob("*.md")) + list((data_dir / "sources").glob("*.md"))

md_documents = [
    Document(
        page_content=path.read_text(encoding="utf-8"),
        metadata={"source": path.name},
    )
    for path in sorted(md_paths)
]

parent_splitter = RecursiveCharacterTextSplitter(chunk_size = 250, chunk_overlap=30)
child_splitter = RecursiveCharacterTextSplitter(chunk_size = 90, chunk_overlap=15)

parent_vector_store = Chroma(
    collection_name="parent_document_md_example",
    embedding_function=embeddings,
)
parent_store = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vector_store,
    docstore=parent_store,
    parent_splitter=parent_splitter,
    child_splitter=child_splitter,
)

parent_retriever.add_documents(md_documents)
results = parent_retriever.invoke("모터 이슈")

for document in results:
    print(len(document.page_content), document.metadata["source"])
    print(document.page_content)
    print("---")

238 PARENT-001.md
# 모터 유지보수 종합 가이드

## 일일 점검

작업 시작 전 모터 외관, 케이블, 이상 소음 여부를 확인합니다. 운전 중에는 진동과 온도 값을 기록하고 이전 점검 기록과 비교합니다.

## 진동 이상

모터 진동이 평소보다 증가하면 먼저 베어링 마모 상태를 확인합니다. 이후 축 정렬 불량, 회전체 불균형, 체결 볼트 풀림 여부를 순서대로 점검합니다. 진동 센서 자체의 체결 상태도 함께 확인해야 합니다.

## 온도 이상
---
154 11_compression_long_report.md
# 모터 설비 월간 점검 보고서

## 생산 현황
이번 달 생산량은 계획 대비 98%였고 설비 가동률은 91%였습니다. 주간 생산 계획에는 큰 차이가 없었습니다.

## 청소 상태
설비 외관 청소 상태는 양호했습니다. 모터 주변 통로에 자재가 일부 적치되어 이동 조치했습니다.
---


In [74]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

base_retriever = vector_store.as_retriever(search_kwargs={"k": 2})
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True,
)

results=multi_query_retriever.invoke("모터에 이상한 소리가 납니다")
for document in results:
    print(document.metadata["document_id"], "|", document.metadata["source"])
    print(document.page_content)

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['모터에서 나는 이상한 소리의 원인은 무엇인가요?  ', '모터 작동 중 발생하는 비정상적인 소리에 대해 알려주세요.  ', '모터에서 소음이 발생할 때 점검해야 할 사항은 무엇인가요?']


R-004 | 04_reorder_documents.csv
모터의 이름판과 정격 전압을 확인합니다.
R-003 | 04_reorder_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬을 우선 점검합니다.
M-002 | 01_common_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
R-008 | 04_reorder_documents.csv
모터 주변 통로의 장애물을 제거합니다.


In [72]:
import logging
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

In [87]:
from uuid import uuid4
from pathlib import Path

from langchain_chroma import Chroma
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.documents import Document
from langchain_core.stores import InMemoryByteStore

md_paths=list(data_dir.glob("*.md")) + list((data_dir / "sources").glob("*.md"))
md_documents = [
    Document(
        page_content=path.read_text(encoding="utf-8"),
        metadata={"source": path.name},
    )
    for path in sorted(md_paths)
]

id_key = "doc_id"
doc_ids = [str(uuid4()) for _ in md_documents]
summary_texts = [
    llm.invoke(f"다음 문서를 한 문장으로 요약해줘:\n\n{doc.page_content}").content
    for doc in md_documents
]

summary_documents = [
    Document(page_content=summary, metadata={id_key: doc_ids[index]})
    for index, summary in enumerate(summary_texts)
]

multi_vector_store = Chroma(
    collection_name="multi_vector_md_example",
    embedding_function=embeddings,
)

byte_store = InMemoryByteStore()
multi_vector_retriever = MultiVectorRetriever(
    vectorstore=multi_vector_store,
    byte_store=byte_store,
    id_key=id_key,
)

multi_vector_retriever.vectorstore.add_documents(summary_documents)
multi_vector_retriever.docstore.mset(list(zip(doc_ids, md_documents)))

results = multi_vector_retriever.invoke("수율, 가동률 저하 원인")
for document in results:
    print(document.metadata["source"])
    print(document.page_content)

11_compression_long_report.md
# 모터 설비 월간 점검 보고서

## 생산 현황
이번 달 생산량은 계획 대비 98%였고 설비 가동률은 91%였습니다. 주간 생산 계획에는 큰 차이가 없었습니다.

## 청소 상태
설비 외관 청소 상태는 양호했습니다. 모터 주변 통로에 자재가 일부 적치되어 이동 조치했습니다.

## 진동 이상
9월 18일 오전부터 모터 진동이 평균 2.2 mm/s에서 6.1 mm/s까지 증가했습니다.
점검 결과 베어링 윤활 상태가 부족했고 축 정렬 오차도 발견되었습니다.
윤활 보충과 축 정렬 작업 후 진동은 2.5 mm/s 수준으로 감소했습니다.

## 온도
진동 증가 시간대에 베어링 온도도 58도에서 76도까지 상승했습니다.
조치 후 온도는 61도로 회복되었습니다.

## 전력
모터 전력 사용량은 전월 대비 2% 증가했으나 생산량 차이를 고려하면 유의한 변화는 아니었습니다.

## 안전
월간 안전 점검에서 비상 정지 버튼과 안전 커버의 이상은 발견되지 않았습니다.

## 향후 조치
진동과 온도가 동시에 증가할 경우 우선 점검 알람을 발생하도록 기준을 검토합니다.

PARENT-001.md
# 모터 유지보수 종합 가이드

## 일일 점검

작업 시작 전 모터 외관, 케이블, 이상 소음 여부를 확인합니다. 운전 중에는 진동과 온도 값을 기록하고 이전 점검 기록과 비교합니다.

## 진동 이상

모터 진동이 평소보다 증가하면 먼저 베어링 마모 상태를 확인합니다. 이후 축 정렬 불량, 회전체 불균형, 체결 볼트 풀림 여부를 순서대로 점검합니다. 진동 센서 자체의 체결 상태도 함께 확인해야 합니다.

## 온도 이상

베어링 온도가 75도를 넘으면 윤활 부족, 냉각 불량, 과부하 여부를 확인합니다. 온도가 계속 상승하면 설비를 정지하고 정비 담당자에게 전달합니다.

## 윤활

윤활유는 6개월 또는 2,000시간 운전 후 교체합니다. 오염도가 높거나 온도 상승이 반복되면 기준 주기보다 빠르게 교체할 수 있습니다.

## 기록

점

In [92]:
!uv add -U langchain-classic langchain-community

Resolved 112 packages in 317ms
Checked 110 packages in 24ms


In [6]:
import csv
from pathlib import Path
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4.1-mini")

data_dir = Path("data/retriever")
target_files = ["01_common_documents.csv", "03_keyword_bm25_documents.csv",
                "04_reorder_documents.csv", "07_selfquery_documents.csv"]

documents = []
for name in target_files:
    file_path = data_dir / name
    with file_path.open(encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            metadata = dict(row)
            metadata.pop("text", None)
            for key in ("year", "priority", "original_order"):
                if key in metadata:
                    metadata[key] = int(metadata[key])
            metadata["source"] = file_path.name
            documents.append(Document(page_content=row["text"], metadata=metadata))

self_query_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="self_query_example",
)
metadata_field_info = [
   AttributeInfo(name="equipment", description="설비 종류 : press, motor, conveyor", type="string"),
    AttributeInfo(name="year", description="문서 작성 연도", type="integer"),
    AttributeInfo(name="priority", description="중요도: 1부터 3 사이의 정수", type="integer"),
]

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=self_query_store,
    document_contents="설비 점검과 안전 조치 메뉴얼",
    metadata_field_info=metadata_field_info,
    enable_limit=True,
)

results = self_query_retriever.invoke(
    "2026년에 작성된 중요도 3의 설비 문서 2개"
)
for document in results:
    print(document.metadata, document.page_content)

ImportError: cannot import name 'DatabricksVectorSearch' from 'langchain_community.vectorstores' (d:\hanwha_0902\rag_two\ex_0923\.venv\Lib\site-packages\langchain_community\vectorstores\__init__.py)

In [90]:
import faiss

from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_community.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_size = len(embeddings.embed_query("벡터 차원 확인"))
index = faiss.IndexFlatL2(embedding_size)
time_vector_store = FAISS(
    embeddings,
    index,
    InMemoryDocstore({}),
    {},
)
time_retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=time_vector_store,
    decay_rate=0.5,
    k=5,
)

time_retriever.add_documents(documents)
results = time_retriever.invoke("모터 유지보수")
for document in results:
    print(document.metadata["document_id"], document.page_content)

R-001 모터 유지보수 시 작업자 보호구 착용 기준을 확인합니다.
R-003 모터 진동 증가 시 베어링 마모와 축 정렬을 우선 점검합니다.
SQ-003 모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
M-002 모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
SQ-004 모터 윤활유는 6개월 또는 2,000시간마다 교체합니다.


d:\hanwha_0902\rag_two\ex_0923\.venv\Lib\site-packages\langchain_classic\retrievers\time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='4cb3534e-9ed2-4d47-8fad-a19840fc562b', metadata={'document_id': 'R-001', 'original_order': 1, 'source': '04_reorder_documents.csv', 'last_accessed_at': datetime.datetime(2026, 9, 23, 12, 39, 2, 486823), 'created_at': datetime.datetime(2026, 9, 23, 12, 39, 2, 486823), 'buffer_idx': 20}, page_content='모터 유지보수 시 작업자 보호구 착용 기준을 확인합니다.'), np.float32(0.48139834)), (Document(id='c3870978-f0ea-45c2-b277-9e9cdcbc2351', metadata={'document_id': 'R-003', 'original_order': 3, 'source': '04_reorder_documents.csv', 'last_accessed_at': datetime.datetime(2026, 9, 23, 12, 39, 2, 486823), 'created_at': datetime.datetime(2026, 9, 23, 12, 39, 2, 486823), 'buffer_idx': 22}, page_content='모터 진동 증가 시 베어링 마모와 축 정렬을 우선 점검합니다.'), np.float32(0.3184647)), (Document(id='37176989-e990-4901-92b5-9fbbfe2b1bf7', metadata={'document_